In [10]:
import pandas as pd
import numpy as np
import re

# Định nghĩa các ký tự tiếng Việt dùng cho Regex
viet_chars = (
    r'àáâãèéêìíòóôõùúýăđơư'
    r'ạảấầẩẫậắằẳẵặẹẻẽếềểễệ'
    r'ỉịọỏốồổỗộớờởỡợụủứừửữự'
    r'ỳỵỷỹÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ'
)

def remove_vietnamese(text):
    if not isinstance(text, str) or pd.isna(text):
        return text
    text = re.sub(rf'\s*\([^)]*[{viet_chars}][^)]*\)', '', text)
    text = re.sub(rf'\s*/[^/]*[{viet_chars}].*', '', text)
    text = re.sub(rf'\s+[{viet_chars}a-zA-Z]*[{viet_chars}][\w\s]*$', '', text)
    return text.strip()

In [11]:
# Chọn 1 bảng bất kỳ để làm Test Data
test_name = "Production_DV_Tile_YB_Powder"

try:
    df_raw_test = pd.read_csv(f"{test_name}.csv")
    print(f" Đọc thành công file Test: {test_name}.csv | Kích thước gốc: {df_raw_test.shape}")
    display(df_raw_test.head(3))
except FileNotFoundError:
    print(f" Không tìm thấy file {test_name}.csv. Vui lòng kiểm tra lại tên file.")

 Đọc thành công file Test: Production_DV_Tile_YB_Powder.csv | Kích thước gốc: (4, 19)


,Production (DV Tile + YB Powder),unit,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD,Q1,Q2,Q3,Q4
0,Production (DV Tile) (Sản phẩm ĐV),ton,9609.931,6310.080933,17732.973467,15795.455367,16091.799767,16408.015833,18454.323805,16179.630248,12136.760367,12464.841216,12324.14555,15580.268823,169088.226375,33652.9854,48295.270967,46770.714419,40369.25559
1,Production (YB Powder) (bột bán cho Yên Bình),ton,0.000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,423.667400,423.667400,0.0000,0.000000,0.000000,423.66740
2,Total Production (DV + YB),ton,9609.931,6310.080933,17732.973467,15795.455367,16091.799767,16408.015833,18454.323805,16179.630248,12136.760367,12464.841216,12324.14555,16003.936223,169511.893775,33652.9854,48295.270967,46770.714419,40792.92299


## Columns Description

---

### 1. Thông tin chung
* **`name`**: Tên sản phẩm. Bao gồm tên tiếng Anh và tiếng Việt (Ví dụ: Coal/ Than 5A).
* **`unit`**: Đơn vị tính. Ở đây là `Ton` (Tấn).
* **`name_no_vn`**: Tên sản phẩm đã lược bỏ tiếng Việt, chỉ giữ lại tiếng Anh hoặc mã hiệu để thuận tiện cho việc xử lý dữ liệu (Data processing).

### 2. Dữ liệu thời gian (Tháng)
Các cột từ `jan` đến `dec` đại diện cho khối lượng tiêu thụ/nhập kho tương ứng với 12 tháng trong năm:
| Cột | Tháng |
| :--- | :--- |
| **jan** | Tháng 1 |
| **feb** | Tháng 2 |
| **mar** | Tháng 3 |
| **apr** | Tháng 4 |
| **may** | Tháng 5 |
| **jun** | Tháng 6 |
| **jul** | Tháng 7 |
| **aug** | Tháng 8 |
| **sep** | Tháng 9 |
| **oct** | Tháng 10 |
| **nov** | Tháng 11 |
| **dec** | Tháng 12 |

### 3. Các ký hiệu đặc biệt trong dữ liệu
* **Số thập phân**: Sử dụng dấu chấm `.` để phân tách (Ví dụ: `1233.522` tấn).
* **`0.000`**: Giá trị bằng không (không có phát sinh trong tháng).
* **`NaN`**: (Not a Number) Dữ liệu trống hoặc chưa có thông tin ghi nhận.
* **Ký hiệu (DV), (YB)**: Đại diện cho các **Cơ sở phát thải (Emission Facilities)** hoặc chi nhánh. ESG yêu cầu báo cáo chi tiết theo từng địa điểm vận hành.

##BƯỚC 1 - Chuẩn hóa Header và Lọc Cột

In [12]:
def clean_headers_and_columns(df):
    df_temp = df.copy()
    if df_temp.iloc[0].astype(str).str.contains('unit', case=False, na=False).any():
        df_temp.columns = [str(c).strip() for c in df_temp.iloc[0]]
        df_temp = df_temp[1:].reset_index(drop=True)

    valid_cols = [c for c in df_temp.columns if not str(c).lower().startswith('unnamed') and str(c).lower() != 'nan']
    df_temp = df_temp[valid_cols]

    cols = list(df_temp.columns)
    if len(cols) >= 2:
        cols[0], cols[1] = "original_name", "unit"
        df_temp.columns = cols
    return df_temp

# --- CHẠY TEST BƯỚC 1 ---
df_step1 = clean_headers_and_columns(df_raw_test)
print(f"--- KẾT QUẢ BƯỚC 1: Kích thước: {df_step1.shape} ---")
display(df_step1.head(10))

--- KẾT QUẢ BƯỚC 1: Kích thước: (4, 19) ---


,original_name,unit,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD,Q1,Q2,Q3,Q4
0,Production (DV Tile) (Sản phẩm ĐV),ton,9609.931,6310.080933,17732.973467,15795.455367,16091.799767,16408.015833,18454.323805,16179.630248,12136.760367,12464.841216,12324.14555,15580.268823,169088.226375,33652.9854,48295.270967,46770.714419,40369.25559
1,Production (YB Powder) (bột bán cho Yên Bình),ton,0.000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,423.667400,423.667400,0.0000,0.000000,0.000000,423.66740
2,Total Production (DV + YB),ton,9609.931,6310.080933,17732.973467,15795.455367,16091.799767,16408.015833,18454.323805,16179.630248,12136.760367,12464.841216,12324.14555,16003.936223,169511.893775,33652.9854,48295.270967,46770.714419,40792.92299
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##BƯỚC 2 - Lọc Rác Theo Dòng

In [13]:
def filter_garbage_rows(df):
    df_temp = df.copy()
    df_temp = df_temp.dropna(subset=['original_name'])
    df_temp = df_temp[df_temp['original_name'].astype(str).str.strip() != '']
    df_temp = df_temp[df_temp['original_name'].astype(str).str.strip().str.lower() != 'nan']
    return df_temp

# --- CHẠY TEST BƯỚC 2 ---
df_step2 = filter_garbage_rows(df_step1)
print(f"--- KẾT QUẢ BƯỚC 2: Kích thước: {df_step2.shape} ---")
display(df_step2.head(10))

--- KẾT QUẢ BƯỚC 2: Kích thước: (3, 19) ---


,original_name,unit,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD,Q1,Q2,Q3,Q4
0,Production (DV Tile) (Sản phẩm ĐV),ton,9609.931,6310.080933,17732.973467,15795.455367,16091.799767,16408.015833,18454.323805,16179.630248,12136.760367,12464.841216,12324.14555,15580.268823,169088.226375,33652.9854,48295.270967,46770.714419,40369.25559
1,Production (YB Powder) (bột bán cho Yên Bình),ton,0.000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,423.667400,423.667400,0.0000,0.000000,0.000000,423.66740
2,Total Production (DV + YB),ton,9609.931,6310.080933,17732.973467,15795.455367,16091.799767,16408.015833,18454.323805,16179.630248,12136.760367,12464.841216,12324.14555,16003.936223,169511.893775,33652.9854,48295.270967,46770.714419,40792.92299


##BƯỚC 3 - Chuyển dọc dữ liệu

In [14]:
def melt_data(df, category_name):
    df_temp = df.copy()
    df_temp.columns = df_temp.columns.astype(str).str.strip()
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    valid_months = [m for m in months if m in df_temp.columns]

    if not valid_months:
        return pd.DataFrame() # Trả về DF rỗng nếu không có cột tháng

    df_long = df_temp.melt(id_vars=["original_name", "unit"], value_vars=valid_months,
                           var_name="period", value_name="value")
    df_long.insert(0, "category", category_name)
    return df_long

# --- CHẠY TEST BƯỚC 3 ---
df_step3 = melt_data(df_step2, test_name)
print(f"--- KẾT QUẢ BƯỚC 3 (MELT): Kích thước: {df_step3.shape} ---")
display(df_step3.head(10))

--- KẾT QUẢ BƯỚC 3 (MELT): Kích thước: (36, 5) ---


,category,original_name,unit,period,value
0,Production_DV_Tile_YB_Powder,Production (DV Tile) (Sản phẩm ĐV),ton,Jan,9609.931000
1,Production_DV_Tile_YB_Powder,Production (YB Powder) (bột bán cho Yên Bình),ton,Jan,0.000000
2,Production_DV_Tile_YB_Powder,Total Production (DV + YB),ton,Jan,9609.931000
3,Production_DV_Tile_YB_Powder,Production (DV Tile) (Sản phẩm ĐV),ton,Feb,6310.080933
4,Production_DV_Tile_YB_Powder,Production (YB Powder) (bột bán cho Yên Bình),ton,Feb,0.000000
5,Production_DV_Tile_YB_Powder,Total Production (DV + YB),ton,Feb,6310.080933
6,Production_DV_Tile_YB_Powder,Production (DV Tile) (Sản phẩm ĐV),ton,Mar,17732.973467
7,Production_DV_Tile_YB_Powder,Production (YB Powder) (bột bán cho Yên Bình),ton,Mar,0.000000
8,Production_DV_Tile_YB_Powder,Total Production (DV + YB),ton,Mar,17732.973467
9,Production_DV_Tile_YB_Powder,Production (DV Tile) (Sản phẩm ĐV),ton,Apr,15795.455367


##BƯỚC 4 - Chuẩn hóa tên (Xóa tiếng Việt)

In [15]:
def normalize_names(df_long):
    df_temp = df_long.copy()
    df_temp.insert(1, "name", df_temp["original_name"].apply(remove_vietnamese))
    rules = {r'Than cám qua sàng / Than ': 'Coal ', r'/ Than non': '', r'/ Than ': ' '}
    for old, new in rules.items():
        df_temp["name"] = df_temp["name"].str.replace(old, new, regex=True)
    df_temp["name"] = df_temp["name"].str.strip()
    return df_temp

# --- CHẠY TEST BƯỚC 4 ---
df_step4 = normalize_names(df_step3)
print("--- KẾT QUẢ BƯỚC 4 ---")
display(df_step4.head(10))

--- KẾT QUẢ BƯỚC 4 ---


,category,name,original_name,unit,period,value
0,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Jan,9609.931000
1,Production_DV_Tile_YB_Powder,Production (YB Powder),Production (YB Powder) (bột bán cho Yên Bình),ton,Jan,0.000000
2,Production_DV_Tile_YB_Powder,Total Production (DV + YB),Total Production (DV + YB),ton,Jan,9609.931000
3,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Feb,6310.080933
4,Production_DV_Tile_YB_Powder,Production (YB Powder),Production (YB Powder) (bột bán cho Yên Bình),ton,Feb,0.000000
5,Production_DV_Tile_YB_Powder,Total Production (DV + YB),Total Production (DV + YB),ton,Feb,6310.080933
6,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Mar,17732.973467
7,Production_DV_Tile_YB_Powder,Production (YB Powder),Production (YB Powder) (bột bán cho Yên Bình),ton,Mar,0.000000
8,Production_DV_Tile_YB_Powder,Total Production (DV + YB),Total Production (DV + YB),ton,Mar,17732.973467
9,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Apr,15795.455367


##BƯỚC 5 - Xử lý số liệu và Ép kiểu

In [16]:
import numpy as np
import pandas as pd

def process_numeric_data(df_long):
    df_temp = df_long.copy()
    df_temp["unit"] = df_temp["unit"].astype(str).str.strip()

    # 1. Chuyển các giá trị trống/không hợp lệ thành NaN
    df_temp["value"] = df_temp["value"].astype(str).str.strip()
    df_temp["value"] = df_temp["value"].str.replace(',', '', regex=False)
    df_temp["value"] = df_temp["value"].replace(['-', '', 'nan', 'NaN', 'None'], np.nan)
    df_temp["value"] = pd.to_numeric(df_temp["value"], errors="coerce")

    # 2. Tạo sẵn dữ liệu cho cột 'has_data' và 'Month'
    has_data_series = df_temp["value"].notna().astype(int)

    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    month_series = df_temp["period"].map(month_map)

    # 3. ĐIỀN TRUNG VỊ (MEDIAN) CỦA RIÊNG TỪNG SẢN PHẨM TRONG 12 THÁNG
    df_temp["value"] = df_temp.groupby("name")["value"].transform(lambda x: x.fillna(x.median()))

    # Đề phòng trường hợp một sản phẩm trống toàn bộ cả 12 tháng (median cũng là NaN) -> điền bằng 0
    df_temp["value"] = df_temp["value"].fillna(0).round(3)

    # 4. CHÈN CỘT VÀO ĐÚNG VỊ TRÍ
    if "period" in df_temp.columns:
        period_idx = df_temp.columns.get_loc("period")
        df_temp.insert(period_idx + 1, "Month", month_series)

    # Chèn cột 'has_data' (1/0) ngay sau cột 'value'
    if "value" in df_temp.columns:
        value_idx = df_temp.columns.get_loc("value")
        df_temp.insert(value_idx + 1, "has_data", has_data_series)

    return df_temp

# --- CHẠY TEST BƯỚC 5 MỚI ---
df_step5 = process_numeric_data(df_step4)
print(f"--- KẾT QUẢ BƯỚC 5: Kích thước: {df_step5.shape} ---")
display(df_step5.head(10))

--- KẾT QUẢ BƯỚC 5: Kích thước: (36, 8) ---


,category,name,original_name,unit,period,Month,value,has_data
0,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Jan,1,9609.931,1
1,Production_DV_Tile_YB_Powder,Production (YB Powder),Production (YB Powder) (bột bán cho Yên Bình),ton,Jan,1,0.000,1
2,Production_DV_Tile_YB_Powder,Total Production (DV + YB),Total Production (DV + YB),ton,Jan,1,9609.931,1
3,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Feb,2,6310.081,1
4,Production_DV_Tile_YB_Powder,Production (YB Powder),Production (YB Powder) (bột bán cho Yên Bình),ton,Feb,2,0.000,1
5,Production_DV_Tile_YB_Powder,Total Production (DV + YB),Total Production (DV + YB),ton,Feb,2,6310.081,1
6,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Mar,3,17732.973,1
7,Production_DV_Tile_YB_Powder,Production (YB Powder),Production (YB Powder) (bột bán cho Yên Bình),ton,Mar,3,0.000,1
8,Production_DV_Tile_YB_Powder,Total Production (DV + YB),Total Production (DV + YB),ton,Mar,3,17732.973,1
9,Production_DV_Tile_YB_Powder,Production (DV Tile),Production (DV Tile) (Sản phẩm ĐV),ton,Apr,4,15795.455,1


##Đóng gói Pipeline (Gom các hàm lại)

In [17]:
def clean_table(df_raw, category_name):
    df1 = clean_headers_and_columns(df_raw)
    df2 = filter_garbage_rows(df1)
    df3 = melt_data(df2, category_name)

    if df3.empty:
        return df3

    df4 = normalize_names(df3)
    df5 = process_numeric_data(df4)
    return df5

print("Đã khởi tạo hàm Pipeline tổng: clean_table()")

Đã khởi tạo hàm Pipeline tổng: clean_table()


##Chạy toàn bộ cho các CSV và xuất Master Data

In [18]:
file_names = ['Raw_Material', 'Production_DV_Tile_YB_Powder']
cleaned_tables = []

print("--- BẮT ĐẦU XỬ LÝ HÀNG LOẠT ---")

for name in file_names:
    try:
        df_raw = pd.read_csv(f"{name}.csv")
        print(f"\nĐang xử lý bảng: {name} (Gốc: {df_raw.shape})")

        # Gọi Pipeline
        df_clean = clean_table(df_raw, name)

        if not df_clean.empty:
            cleaned_tables.append(df_clean)
            print(f"  -> Thành công! Kích thước sạch: {df_clean.shape}")
        else:
            print("  -> Bỏ qua vì không có cột tháng hợp lệ.")

    except FileNotFoundError:
        print(f" Bỏ qua: Không tìm thấy {name}.csv")

# Gộp bảng
if cleaned_tables:
    final_df = pd.concat(cleaned_tables, ignore_index=True)

    # Cập nhật danh sách các cột để bao gồm 'Month' và 'has_data'
    final_cols = ['category', 'name', 'original_name', 'unit', 'period', 'Month', 'value', 'has_data']

    # Kiểm tra xem các cột có thực sự tồn tại trước khi lọc để tránh lỗi
    existing_cols = [col for col in final_cols if col in final_df.columns]
    final_df = final_df[existing_cols]

    # Xuất file CSV
    final_df.to_csv("Production_Raw_Mat_Cleaned.csv", index=False)

    print(f"\n XONG! Đã lưu file Master (Tổng dòng: {len(final_df)}).")
    display(final_df.sample(20))
else:
    print("\n KHÔNG CÓ DỮ LIỆU ĐỂ GỘP! Vui lòng kiểm tra lại file CSV gốc.")

--- BẮT ĐẦU XỬ LÝ HÀNG LOẠT ---

Đang xử lý bảng: Raw_Material (Gốc: (14, 19))
  -> Thành công! Kích thước sạch: (168, 8)

Đang xử lý bảng: Production_DV_Tile_YB_Powder (Gốc: (4, 19))
  -> Thành công! Kích thước sạch: (36, 8)

 XONG! Đã lưu file Master (Tổng dòng: 204).


,category,name,original_name,unit,period,Month,value,has_data
125,Raw_Material,% Alternative raw materials,% Alternative raw materials,%,Sep,9,0.070,1
134,Raw_Material,Pottery Stone,Pottery Stone,ton,Oct,10,0.000,0
73,Raw_Material,Gypsum,Gypsum/ Thạch cao,ton,Jun,6,0.000,0
10,Raw_Material,Recycled Mat,Recycled Mat/ Nguyên liệu tái sử dụng (gạch tá...,ton,Jan,1,625.130,1
199,Production_DV_Tile_YB_Powder,Production (YB Powder),Production (YB Powder) (bột bán cho Yên Bình),ton,Nov,11,0.000,0
138,Raw_Material,Total raw materials,Total raw materials/ Tổng nguyên liệu,ton,Oct,10,14137.845,1
44,Raw_Material,Sand,Sand/ Cát,ton,Apr,4,0.000,0
176,Production_DV_Tile_YB_Powder,Total Production (DV + YB),Total Production (DV + YB),ton,Mar,3,17732.973,1
45,Raw_Material,Gypsum,Gypsum/ Thạch cao,ton,Apr,4,0.000,0
83,Raw_Material,% Alternative raw materials,% Alternative raw materials,%,Jun,6,0.071,1
